In [1]:
import torch
from diffusers import DDPMScheduler, UNet2DModel
from torchvision import transforms, datasets
import matplotlib.pyplot as plt
import numpy as np

# 1. Load a Pre-trained Diffusion Model (CIFAR-10)
# We use a standard DDPM model to match the paper's experiment
model_id = "google/ddpm-cifar10-32"
model = UNet2DModel.from_pretrained(model_id).to("cuda" if torch.cuda.is_available() else "cpu")
scheduler = DDPMScheduler.from_pretrained(model_id)
model.eval()

def compute_vlb_proxy(model, image_tensor, timesteps=10):
    """
    Computes a proxy for the Variational Lower Bound (VLB).
    In diffusion, we approximate log-likelihood by measuring the 
    reconstruction error (epsilon prediction) across timesteps.
    """
    device = next(model.parameters()).device
    image_tensor = image_tensor.to(device)
    total_loss = 0
    
    with torch.no_grad():
        for t in np.linspace(0, scheduler.config.num_train_timesteps - 1, timesteps).astype(int):
            t_tensor = torch.tensor([t], device=device)
            # Add noise to the image
            noise = torch.randn_like(image_tensor)
            noisy_image = scheduler.add_noise(image_tensor, noise, t_tensor)
            
            # Predict the noise
            model_output = model(noisy_image, t_tensor).sample
            
            # The VLB is inversely proportional to the MSE of the noise prediction
            # Higher likelihood = Lower noise prediction error
            loss = torch.nn.functional.mse_loss(model_output, noise)
            total_loss += loss.item()
            
    # We return negative loss to represent Log-Likelihood (higher is better)
    return -total_loss

# 2. Prepare Sample Data
# Using a transform to match CIFAR-10 training normalization [-1, 1]
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5]) 
])
dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
sample_image, _ = dataset[0] # Take the first training image

# 3. Replicate Figure 2 Experiment
def run_replication(num_subplots=5):
    epsilons = np.linspace(-0.2, 0.2, 21) # Larger range for visualization
    fig, axes = plt.subplots(1, num_subplots, figsize=(18, 4))
    
    print("Running pixel perturbations...")
    for i in range(num_subplots):
        # Randomly select a pixel (C, Y, X)
        c, y, x = np.random.randint(0, 3), np.random.randint(0, 32), np.random.randint(0, 32)
        scores = []
        
        for eps in epsilons:
            perturbed = sample_image.clone()
            perturbed[c, y, x] += eps # Perturb the input along one dimension
            
            score = compute_vlb_proxy(model, perturbed.unsqueeze(0))
            scores.append(score)
            
        # Plotting the "Concave" pattern
        axes[i].plot(epsilons, scores, marker='o', color='tab:blue', markersize=4, label='Perturbed (Blue)')
        # Mark original training value at eps=0 in orange
        center_idx = len(epsilons) // 2
        axes[i].scatter(0, scores[center_idx], color='tab:orange', s=100, zorder=5, label='Original (Orange)')
        
        axes[i].set_title(f"Pixel ({y}, {x}) Ch {c}")
        axes[i].set_xlabel("Perturbation")
        if i == 0: axes[i].set_ylabel("Proxy Log-Likelihood")

    plt.suptitle("Replication of Figure 2: Training Data as Local Maximum")
    plt.tight_layout()
    plt.show()



/Users/skumar/Desktop/DA5001/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/skumar/Desktop/DA5001/.venv/lib/python3.14/site-packages/diffusers/models/transformers/transformer_kandinsky.py:168: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  @torch.autocast(device_type="cuda", dtype=torch.float32)
/Users/skumar/Desktop/DA5001/.venv/lib/python3.14/site-packages/diffusers/models/transformers/transformer_kandinsky.py:272: UserWarning: CUDA is not available or torch_xla is imported. Disabling autocast.
  @torch.autocast(device_type="cuda", dtype=torch.float32)
/Users/skumar/Desktop/DA5001/.venv/lib/python3.14/site-packages/huggingface_hub/utils/_validators.py:202: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `hf_hub_download`

URLError: <urlopen error [SSL: CERTIFICATE_VERIFY_FAILED] certificate verify failed: unable to get local issuer certificate (_ssl.c:1081)>

In [ ]:
run_replication()